What are the uncertainties I get on the mass fluxes by solving this linear system?

$$SHF + c_p~m_{E} \cdot (\theta_E - \theta_{BL}) + c_p~m_{cd} \cdot (\theta_{cd} - \theta_{BL}) + R + c_p~\rho~H~F_{a,\theta} = 0$$
$$LHF + L_v~m_{E} \cdot (q_E - q_{BL}) + L_v~m_{cd} \cdot (q^*_{cd} - q_{BL}) + L_v~\rho~H~F_{a,q} = 0$$

In [3]:
## what is the reference layer to compute mixed layer AVERAGED quantities?
mixed_avg_levels   = slice(200, 500)

ref_bl_hgt = 500.      # m
ref_bl_density = 1.    # kg / m3

## what height is entrained air from?
entrainment_levels = slice(700, 1000)    # m

## what height does convective downdrafts come from?
downdraft_levels_eu = slice(1000, 2000)
downdraft_levels_or = slice(2500, 3500)

Rd = 287.
cp = 1004. # j / kg K
Lv = 2.5e6 # J/kg
g = 9.81   # m/s2

In [ ]:
import numpy as np
import xarray as xr
from scipy.optimize import root_scalar

B = 17.502 # 1
C = 240.97 # °C
epsilon = 0.622
    
def e_sat(t):
    # t in K
    e_sat0 = 6.1121 # hPa
    return e_sat0 * np.exp( B*(t-273.15) / (C+(t-273.15) ))

def q_sat(t,p):
    # t in K
    # p in hPa  
    return epsilon*e_sat(t)/(p-e_sat(t))
    
def dq_sat_dt(t,p):
    # t in K
    # p in hPa 
    return epsilon*e_sat(t)*p*B*C/((p-e_sat(t))**2*(C+(t-273.15))**2)



## TO USE IN SOLVER find_theta_for_pair
## h_{cd} - s_{cd} - Lv*q_sat(s_{cd}/cp) = 0
## where I find theta as cp*\theta = s_cd
def temp_from_h(theta, p, h_cd, Lv, cp, sat_frac):
    # input:
    # - p : pressure in hPa
    # - h_cd : moist static energy at convective downdraft level, J/kg
    # - Lv : latent heat of vaporization = 2.5e6 J/kg
    # - cp : specific heat dry air = 1004.67 J/kg K
    # - sat_frac : fraction of saturation, to test when different from 1.
    T = theta*(p/1000.)**(Rd/cp)
    # es = e_sat(T)    #6.1121 * np.exp(17.502 * ( T -273.15) / (240.97 + ( T -273.15)))
    # denominator = p - 0.378 * es * (1.0007 + p * 3.46e-6)
    # q_sat = q_sat(T, p) #0.622*es*(1.0007 + p * 3.46e-6) / denominator
    
    return h_cd/cp - theta - sat_frac*(Lv / cp)*q_sat(T,p)


def find_theta_for_pair(p, h, sat_frac):
    
    f = lambda t : temp_from_h(t, p=p, h_cd=h, Lv=Lv, cp=cp, sat_frac=sat_frac)

    try:
        sol = root_scalar(f, bracket=[270,350], method='brentq')
        return sol.root if sol.converged else np.nan
    except:
        return np.nan
    
# Vectorize the function
vectorized_theta_root = np.vectorize(find_theta_for_pair)


## this function aims at extracting the thermodynamical properties 
## from entrainment heights and downdrafts, from a single profile at a time
def properties_from_profile(profile, mixed_avg_levels, entrainment_levels, downdraft_levels, CD_sat_frac, vert_dim):

    h_cd = (cp*profile["ta"] + Lv*profile["q"] + g*profile[vert_dim]).sel({vert_dim:downdraft_levels})
    p_cd = profile["p"].sel({vert_dim:downdraft_levels}).values/100.  ## hPa
    sfc_p = 1000*np.ones_like(p_cd)

    cd_levels = profile[vert_dim].sel({vert_dim:downdraft_levels}).values

    thetaD = (vectorized_theta_root( sfc_p , h_cd, sat_frac=CD_sat_frac))
    thetaD = xr.DataArray(thetaD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_cd)})
    thetaD = thetaD.rename({vert_dim:"height_cd"})

    qD = CD_sat_frac*q_sat(thetaD, sfc_p)  #(meteo.qsea((thetaD)-273.15 , sfc_p)/1e3)
    qD = xr.DataArray(qD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_cd)})
    qD = qD.rename({vert_dim:"height_cd"})

    theta_out = profile.theta.sel({vert_dim:entrainment_levels})  #.rename({vert_dim:"height_e"})
    theta_bl  = profile.theta.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim) #.rename({vert_dim:"height_e"})

    q_out = profile.q.sel({vert_dim:entrainment_levels})  #.rename({vert_dim:"height_e"})
    q_bl  = profile.q.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim) #.rename({vert_dim:"height_e"})


    return xr.Dataset(
        dict(
            thetaD=thetaD,
            qD=qD,
            theta_out=theta_out,
            theta_bl=theta_bl,
            q_out=q_out,
            q_bl=q_bl
        )
        )



## input the properties_from_profile() results
## build the matrix with due coefficients
## to solve for the entrainment and CD mass fluxes
def solve_qTh_equations(profile_props, shf, lhf, RNet, Fa_q, Fa_th):

    # size = (Elevs, 2, 2)
    A = np.zeros((2, 2))
    A[0, 0] = (profile_props["theta_out"].mean() - profile_props["theta_bl"]).values
    A[0, 1] = (profile_props["thetaD"].weighted((profile_props["thetaD"].p_cd).diff(dim="height_cd", n=1)).mean() - profile_props["theta_bl"]).values
    A[1, 0] = (profile_props["q_out"].mean() - profile_props["q_bl"]).values
    A[1, 1] = (profile_props["qD"].weighted((profile_props["thetaD"].p_cd).diff(dim="height_cd", n=1)).mean() - profile_props["q_bl"]).values

    ## DEBUG
    det = np.linalg.det(A)

    # shape (Elevs, 2)
    b = np.zeros((2))
    b[0] = - shf / cp - (RNet) / cp - ref_bl_density*ref_bl_hgt*Fa_th
    b[1] = - lhf / Lv - ref_bl_density*ref_bl_hgt*Fa_q

    # results shape (Elevs, 2)
    x = np.linalg.solve(A, b)

    me = xr.DataArray(x[0], attrs={"units":"kg / m2 s"})
    mD = xr.DataArray(x[1], attrs={"units":"kg / m2 s"})

    m = xr.Dataset(
        dict(
            me = me, 
            mD = mD
        )
    )

    return m, A, det





def props_and_uncertainties_from_profile(ds, downdraft_levels, entrainment_levels, CD_sat_frac, vert_dim, time_dim):
    mse = (cp*ds["ta"] + Lv*ds["q"] + g*ds[vert_dim]).sel({vert_dim:downdraft_levels})
    mse_mean = (mse).mean(dim=time_dim)
    mse_std  = (mse).std(dim=time_dim)
    p_mean = ds["p"].sel({vert_dim:downdraft_levels}).mean(dim=time_dim).values/100.  ## hPa
    sfc_p = 1000*np.ones_like(p_mean)

    cd_levels = ds[vert_dim].sel({vert_dim:downdraft_levels}).values

    thetaD = (vectorized_theta_root( sfc_p , mse_mean, sat_frac=CD_sat_frac))
    qD = CD_sat_frac*q_sat(thetaD, sfc_p)

    # uncertainty on thetaD to be estimated from the variability of MSE
    # dq*_dTheta \approx dq*_dT
    u_thetaD = mse_std / ( cp + Lv*dq_sat_dt(thetaD, p_mean) )
    u_qD     = dq_sat_dt(thetaD, p_mean) * u_thetaD

    thetaD = xr.DataArray(thetaD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_mean)})
    thetaD = thetaD.rename({vert_dim:"height_cd"})
    qD = xr.DataArray(qD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_mean)})
    qD = qD.rename({vert_dim:"height_cd"})

    u_thetaD = xr.DataArray(u_thetaD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_mean)})
    u_thetaD = u_thetaD.rename({vert_dim:"height_cd"})
    u_qD = xr.DataArray(u_qD, dims=[vert_dim], coords={vert_dim:cd_levels, "p_cd": (vert_dim, p_mean)})
    u_qD = u_qD.rename({vert_dim:"height_cd"})


    # same for entrainment
    theta_out = ds.theta.sel({vert_dim:entrainment_levels}).mean(dim=time_dim)  #.rename({vert_dim:"height_e"})
    u_theta_out = ds.theta.sel({vert_dim:entrainment_levels}).std(dim=time_dim)
    theta_bl  = ds.theta.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim).mean(dim=time_dim) #.rename({vert_dim:"height_e"})
    u_theta_bl = ds.theta.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim).std(dim=time_dim)

    q_out = ds.q.sel({vert_dim:entrainment_levels}).mean(dim=time_dim)  #.rename({vert_dim:"height_e"})
    u_q_out = ds.q.sel({vert_dim:entrainment_levels}).std(dim=time_dim)
    q_bl  = ds.q.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim) #.rename({vert_dim:"height_e"})
    u_q_bl = ds.q.sel({vert_dim:mixed_avg_levels}).mean(dim = vert_dim).std(dim=time_dim)


    return xr.Dataset(
        dict(
            thetaD=thetaD,
            u_thetaD=u_thetaD,
            qD=qD,
            u_qD=u_qD,
            theta_out=theta_out,
            u_theta_out=u_theta_out,
            theta_bl=theta_bl,
            u_theta_bl=u_theta_bl,
            q_out=q_out,
            u_q_out=u_q_out,
            q_bl=q_bl,
            u_q_bl=u_q_bl
        )
        )



In [12]:
import eurec4a

cat = eurec4a.get_intake_catalog(use_ipfs="QmahMN2wgPauHYkkiTGoG2TpPBmj3p5FoYJAq9uE9iXT9N")

ds = cat.radiosondes.bco.to_dask().load()

min_hgt = 40.

# level2_path = "/work/mh1498/m301248/TCO_data/eurec4a_profiles_level2/EUREC4A_BCO_Vaisala-RS_L2_v3.0.0.nc"
# level2_path_local = "/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/radiosondes_3/EUREC4A_BCO_Vaisala-RS_L2_v3.0.0.nc"

# ds_eureca = xr.open_dataset(level2_path_local)
ds_eureca_bco = ds.where(((ds["platform"] == 1)&(ds["ascent_flag"]==1)&(ds.alt>min_hgt)), drop=True)
## renaming for common ground
ds_eureca_bco = ds_eureca_bco.swap_dims({"sounding": "launch_time"})
ds_eureca_bco = ds_eureca_bco.rename({"alt":"height"})


ds_orcestra = xr.open_dataset("ipfs://bafybeid7cnw62zmzfgxcvc6q6fa267a7ivk2wcchbmkoyk4kdi5z2yj2w4", engine="zarr")
ds_orcestra
ds_orcestra_bco = ds_orcestra.where(((ds_orcestra["platform"] == "BCO")&(ds_orcestra["ascent_flag"]==0)&(ds_orcestra.height>min_hgt)), drop=True)


/Users/alessandrostorer/.local/share/mamba/envs/env_hacka/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [13]:
props_eu = props_and_uncertainties_from_profile(ds_eureca_bco, downdraft_levels_eu, entrainment_levels, 1., "height", "launch_time")
props_eu

<xarray.Dataset> Size: 8kB
Dimensions:      (height_cd: 101, height: 31, launch_time: 182)
Coordinates:
  * height_cd    (height_cd) int16 202B 1000 1010 1020 1030 ... 1980 1990 2000
    p_cd         (height_cd) float32 404B 905.8 904.7 903.7 ... 806.3 805.3
  * height       (height) int16 62B 700 710 720 730 740 ... 960 970 980 990 1000
  * launch_time  (launch_time) datetime64[ns] 1kB 2020-01-16T20:53:46 ... 202...
    sounding     (launch_time) object 1kB 'BCO__ascent__13.16_-59.43__2020011...
Data variables:
    thetaD       (height_cd) float64 808B 293.6 293.6 293.5 ... 290.9 290.8
    u_thetaD     (height_cd) float64 808B 0.906 0.9085 0.9098 ... 2.046 2.05
    qD           (height_cd) float64 808B 0.0153 0.01528 ... 0.01286 0.01283
    u_qD         (height_cd) float64 808B 0.0009728 0.0009754 ... 0.002123
    theta_out    (height) float32 124B 298.8 298.9 298.9 ... 299.8 299.8 299.9
    u_theta_out  (height) float32 124B 0.5338 0.5432 0.555 ... 0.722 0.7174
    theta_bl     float64 8B 298.4
    u_theta_bl   float64 8B 0.3852
    q_out        (height) float32 124B 0.01418 0.0141 ... 0.01227 0.01221
    u_q_out      (height) float32 124B 0.00102 0.001066 ... 0.001389 0.0014
    q_bl         (launch_time) float32 728B 0.01515 0.01526 ... 0.01459 0.01566
    u_q_bl       float64 8B 0.0009372